# Alpamayo 1.5: Offline Local Inference with Navigation Input

This notebook extends offline local trajectory inference with optional natural-language navigation conditioning.

Features:
1. Supports both single-window and sliding-window offline inference
2. Adds optional `nav_text` input to the trajectory prediction prompt
3. Displays the active navigation text in console output, figure titles, and summary tables
4. Uses one unified navigation text for all windows in sliding mode
5. Prints compact trajectory / metric outputs per window


In [ ]:
import os
import sys
import textwrap
from pathlib import Path

repo_root = Path.cwd().resolve().parent
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import av
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import BitsAndBytesConfig

from alpamayo1_5 import helper
from alpamayo1_5.load_offline_dataset import (
    load_offline_dataset,
    get_or_build_offline_clip_cache,
)
from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5
from alpamayo1_5.offline_eval_utils import (
    set_reproducible_seeds,
    rotate_xyz_xy_plane,
    run_offline_inference_window,
    _compute_adaptive_xy_limits,
)

print('repo_root =', repo_root)
print('src_path =', src_path)


In [ ]:
# ===== Reproducibility =====
SEED = 42
set_reproducible_seeds(SEED)

# ===== Paths =====
CLIP_DIR = Path('/workspace/dataset/')
MODEL_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Alpamayo-1.5-10B/snapshots/f11cd25b758ab560114019b555dde2a8b92d88b4')
PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--Qwen--Qwen3-VL-8B-Instruct/snapshots/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b')
COSMOS_PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Cosmos-Reason2-8B/snapshots/f715d875a8a99a0a2b65aa28633afd9417e46bd9')

# ===== Inference config =====
DEVICE = 'cuda'
NUM_HISTORY_STEPS = 16
NUM_FUTURE_STEPS = 64
TIME_STEP = 0.1
NUM_FRAMES = 4
FPS = 30.0
FRAME0_GPS_TIME_SOD = 175484.98
NUM_TRAJ_SAMPLES = 1
TEMPERATURE = 0.6
TOP_P = 0.98
MAX_GENERATION_LENGTH = 256
IMAGE_SIZE = (448, 800)
EVAL_XY_ROTATION_DEG = -90.0

# ===== Run mode =====
RUN_MODE = 'single'  # 'single' or 'sliding'
SINGLE_T0_SOD = 175500.23
SLIDING_T0_SOD_LIST = []
SLIDING_STEP_SEC = 1.0
MAX_SLIDING_WINDOWS = None

# ===== Navigation config =====
ENABLE_NAV_INPUT = True
NAV_TEXT = 'Turn right in 30m'
NAV_TEXT_MAX_DISPLAY_CHARS = 120

os.environ['ALPAMAYO_VLM_PROCESSOR_PATH'] = str(COSMOS_PROCESSOR_PATH)

print('SEED =', SEED)
print('RUN_MODE =', RUN_MODE)
print('CLIP_DIR =', CLIP_DIR)
print('ENABLE_NAV_INPUT =', ENABLE_NAV_INPUT)
print('NAV_TEXT =', NAV_TEXT)


### Helper functions


In [ ]:
def resolve_nav_text(enable_nav_input: bool, nav_text: str | None) -> str | None:
    if not enable_nav_input:
        return None
    if nav_text is None:
        return None
    nav_text = str(nav_text).strip()
    return nav_text if nav_text else None


def format_nav_text_for_display(nav_text: str | None, max_chars: int = 120) -> str:
    if nav_text is None:
        return '<none>'
    text = str(nav_text).strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + ' ...'


def extract_minade_text(df_metrics: pd.DataFrame) -> str:
    if df_metrics is None or len(df_metrics) == 0:
        return 'N/A'

    row = df_metrics.iloc[0]
    candidate_keys = ['minADE_m', 'minADE', 'min_ade', 'ADE', 'ade']
    for key in candidate_keys:
        if key in row.index:
            value = row[key]
            if pd.notna(value):
                return f'{float(value):.4f}'
    return 'N/A'


def format_cot_text(cot_value, minade_text, nav_text_display, wrap_width=72, max_chars=800):
    cot_part = 'Chain-of-Causation: <empty>' if cot_value is None else str(cot_value).strip()
    if len(cot_part) > max_chars:
        cot_part = cot_part[:max_chars] + ' ...'
    cot_wrapped = textwrap.fill(cot_part, width=wrap_width)
    return (
        f'nav: {nav_text_display}\n'
        f'minADE: {minade_text}\n\n'
        f'Chain-of-Causation:\n{cot_wrapped}'
    )


def compute_max_valid_t0_range(cache, *, num_history_steps, num_future_steps, time_step, num_frames, fps, frame0_gps_time_sod):
    pose_min = float(cache.pose_times_sod[0])
    pose_max = float(cache.pose_times_sod[-1])

    history_left_span = (num_history_steps - 1) * time_step
    future_right_span = num_future_steps * time_step

    pose_t0_min = pose_min + history_left_span
    pose_t0_max = pose_max - future_right_span

    image_left_span = (num_frames - 1) * time_step
    image_t0_min = frame0_gps_time_sod + image_left_span
    return pose_t0_min, pose_t0_max, image_t0_min


def build_t0_list(run_mode, single_t0_sod, sliding_t0_sod_list, cache, *, num_history_steps, num_future_steps, time_step, num_frames, fps, frame0_gps_time_sod, max_windows=None, sliding_step_sec=1.0):
    if run_mode == 'single':
        return [float(single_t0_sod)]

    if sliding_t0_sod_list:
        t0_list = [float(x) for x in sliding_t0_sod_list]
    else:
        pose_t0_min, pose_t0_max, image_t0_min = compute_max_valid_t0_range(
            cache,
            num_history_steps=num_history_steps,
            num_future_steps=num_future_steps,
            time_step=time_step,
            num_frames=num_frames,
            fps=fps,
            frame0_gps_time_sod=frame0_gps_time_sod,
        )

        t0_min = max(pose_t0_min, image_t0_min)
        t0_max = pose_t0_max

        if t0_max < t0_min:
            raise ValueError(f'No valid sliding t0 range. Computed range: [{t0_min:.3f}, {t0_max:.3f}]')

        t0_list = list(np.arange(t0_min, t0_max + 1e-9, sliding_step_sec, dtype=np.float64))

    if max_windows is not None:
        t0_list = t0_list[:max_windows]
    return t0_list


def make_compact_bev_table(result):
    pd.set_option('display.float_format', lambda x: f'{x:.6f}')
    gt = pd.DataFrame(result['gt_xyz_plot'], columns=['gt_x', 'gt_y', 'gt_z'])
    pred = pd.DataFrame(result['pred_best_xyz'], columns=['pred_x', 'pred_y', 'pred_z'])
    n = max(len(gt), len(pred))
    df = pd.DataFrame({'step': np.arange(n), 't_sec': np.arange(n) * TIME_STEP})
    return df.join(gt).join(pred)


def render_result_figure(result, nav_text_display: str):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    cot_text = format_cot_text(
        result.get('cot_value', ''),
        minade_text=extract_minade_text(result.get('df_metrics')),
        nav_text_display=nav_text_display,
    )

    ax = axes[0]
    ax.text(
        0.02, 0.98, cot_text,
        transform=ax.transAxes,
        va='top', ha='left', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='whitesmoke', alpha=0.9),
    )
    ax.axis('off')
    ax.set_title('Navigation / CoT / Metrics')

    ax = axes[1]
    hist_xyz = result['hist_xyz_plot']
    gt_xyz = result['gt_xyz_plot']
    pred_xyz = result['pred_best_xyz']

    xlim, ylim = _compute_adaptive_xy_limits(hist_xyz, gt_xyz, pred_xyz, min_span=20.0, pad_ratio=0.12, pad_min=2.0)

    ax.plot(hist_xyz[:, 0], hist_xyz[:, 1], 'b-o', linewidth=2, markersize=3, label='History')
    ax.plot(gt_xyz[:, 0], gt_xyz[:, 1], 'k-o', linewidth=2.5, markersize=3.5, label='GT')
    ax.plot(pred_xyz[:, 0], pred_xyz[:, 1], 'r-o', linewidth=2.5, markersize=3.5, label='Pred')
    ax.scatter([0.0], [0.0], c='red', marker='x', s=120, label='t0 ego')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'Offline trajectory inference | nav={nav_text_display}')
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal', adjustable='box')
    ax.legend(loc='best')

    plt.tight_layout()
    return fig


def run_navigation_window(t0_sod, *, nav_text, model, processor):
    data = load_offline_dataset(
        clip_dir=CLIP_DIR,
        t0_sod=float(t0_sod),
        num_history_steps=NUM_HISTORY_STEPS,
        num_future_steps=NUM_FUTURE_STEPS,
        time_step=TIME_STEP,
        num_frames=NUM_FRAMES,
        fps=FPS,
        frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
        debug=False,
        image_size=IMAGE_SIZE,
    )

    result = run_offline_inference_window(
        data=data,
        model=model,
        processor=processor,
        device=DEVICE,
        top_p=TOP_P,
        temperature=TEMPERATURE,
        num_traj_samples=NUM_TRAJ_SAMPLES,
        max_generation_length=MAX_GENERATION_LENGTH,
        time_step=TIME_STEP,
        eval_xy_rotation_deg=EVAL_XY_ROTATION_DEG,
        nav_text=nav_text,
    )

    metrics_df = result.get('df_metrics')
    metrics_row = metrics_df.iloc[0].to_dict() if metrics_df is not None and len(metrics_df) else {}
    nav_text_display = format_nav_text_for_display(nav_text, max_chars=NAV_TEXT_MAX_DISPLAY_CHARS)

    return {
        't0_sod': float(t0_sod),
        'nav_text': nav_text,
        'nav_text_display': nav_text_display,
        'result': result,
        'metrics_row': metrics_row,
    }


### Load cache, model, and processor


In [ ]:
clip_cache = get_or_build_offline_clip_cache(CLIP_DIR, debug=False, force_rebuild=False)

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False,
)

model = Alpamayo1_5.from_pretrained(
    str(MODEL_PATH),
    dtype=torch.bfloat16,
    device_map='cuda:0',
    quantization_config=quantization_config,
)

processor = helper.get_processor(
    model.tokenizer,
    processor_name_or_path=PROCESSOR_PATH,
)

print('Loaded cache, model, and processor.')


### Run offline local inference with optional navigation input


In [ ]:
resolved_nav_text = resolve_nav_text(ENABLE_NAV_INPUT, NAV_TEXT)
print('Resolved nav_text =', resolved_nav_text)

t0_list = build_t0_list(
    run_mode=RUN_MODE,
    single_t0_sod=SINGLE_T0_SOD,
    sliding_t0_sod_list=SLIDING_T0_SOD_LIST,
    cache=clip_cache,
    num_history_steps=NUM_HISTORY_STEPS,
    num_future_steps=NUM_FUTURE_STEPS,
    time_step=TIME_STEP,
    num_frames=NUM_FRAMES,
    fps=FPS,
    frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
    max_windows=MAX_SLIDING_WINDOWS,
    sliding_step_sec=SLIDING_STEP_SEC,
)

print('num windows =', len(t0_list))
print('first few t0_sod =', t0_list[:5])
if len(t0_list):
    print('last few t0_sod =', t0_list[-5:])

window_results = []

for i, t0_sod in enumerate(t0_list, start=1):
    print(f'\n=== Running window {i}/{len(t0_list)} | t0_sod={t0_sod:.3f} ===')
    print('nav_text =', format_nav_text_for_display(resolved_nav_text, max_chars=NAV_TEXT_MAX_DISPLAY_CHARS))

    wr = run_navigation_window(
        t0_sod=t0_sod,
        nav_text=resolved_nav_text,
        model=model,
        processor=processor,
    )
    window_results.append(wr)

    metrics_row = wr['metrics_row']
    print(
        f"best_sample_idx={metrics_row.get('best_sample_idx', 'N/A')}, "
        f"minADE_m={metrics_row.get('minADE_m', 'N/A')}, "
        f"final_point_error_m={metrics_row.get('final_point_error_m', 'N/A')}"
    )

    fig = render_result_figure(wr['result'], wr['nav_text_display'])
    plt.show()
    plt.close(fig)

    compact_df = make_compact_bev_table(wr['result'])
    print(f"[Window t0_sod={wr['t0_sod']:.6f}] GT / Pred trajectory table | nav={wr['nav_text_display']}")
    display(compact_df)

print('\nFinished all windows.')


### Concise result summary table


In [ ]:
summary_rows = []
for wr in window_results:
    row = {
        't0_sod': wr['t0_sod'],
        'nav_text': wr['nav_text_display'],
    }
    row.update(wr['metrics_row'])
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
display(df_summary)
